# Prerequisites
본 `ipynb` 은 `Python=3.12` 에서 작성하였습니다. Package dependency 를 해결하기 위해 아래 cell 을 실행해주세요.

## Install Python packages

In [ ]:
%pip -q install -U openai==2.36.0 "openai[realtime]" sounddevice==0.5.5 dotenv==0.9.9 azure-ai-voicelive==1.2.0 websocket-client==1.9.0 aiohttp==3.14.0

## Load environment variables from a .env file
secret 노출을 피하고 notebook 들간의 일관된 환경변수를 설정하기 위해 `dotenv` 을 이용한다.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_OPENAI_URI = os.getenv("AZURE_OPENAI_URI")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_CHAT_DEPLOYMENT = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
AZURE_OPENAI_REALTIME_DEPLOYMENT = os.getenv("AZURE_OPENAI_REALTIME_DEPLOYMENT")
AZURE_SPEECH_LEXICON_URL = os.getenv("AZURE_SPEECH_LEXICON_URL") # 각자의 lexicon.xml URL을 .env 파일에 설정해주세요.

# Handling session expire of gpt-realtime

`001-model-005-voice-basic.ipynb` 의 gpt-realtime 데모를 확장해, **세션 최대 지속시간(`expires_at`) 제한**을 다루는 방식을 정리한다.

Realtime 세션은 `session.created` 이벤트의 `expires_at`(Unix epoch, 초) 시각에 서버가 **강제로 연결을 끊는다.** 이 시간은 서버측 제한이다. Session 의 extend 는 미지원하므로, 끊김 없는 장시간 대화가 목표라면 **② 재연결 + 컨텍스트 복원** 을 수행한다.

참고: [Azure OpenAI Realtime audio — session timeout](https://learn.microsoft.com/ko-kr/azure/foundry/openai/how-to/realtime-audio)

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_OPENAI_URI = os.getenv("AZURE_OPENAI_URI")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_REALTIME_DEPLOYMENT = os.getenv("AZURE_OPENAI_REALTIME_DEPLOYMENT")

In [ ]:
import asyncio
import base64
import threading
import time

import sounddevice as sd
from openai import AsyncOpenAI

# ══════════════════════════════════════════════════════════════════════════
# 방식 ① session.update keep-alive — 만료 직전 session.update 재전송으로 연장 시도
# ⚠️ expires_at 은 하드 리밋이라 항상 연장된다는 보장은 없다(연장 안 되면 그대로 만료).
# ══════════════════════════════════════════════════════════════════════════

# 재실행 시 남아 있는 PortAudio 상태 초기화 (-9986 예방)
def reset_portaudio():
    try:
        sd._terminate()
    except Exception as e:
        print(f"PortAudio reset error: {e}")
    sd._initialize()

# 지터 버퍼 스피커 (재생 끊김 완화)
class Speaker:
    def __init__(self, samplerate=24000, prebuffer_ms=300):
        self.sr = samplerate
        self.buf = bytearray()
        self.pos = 0
        self.lock = threading.Lock()
        self.playing = False
        self.prebuffer = int(samplerate * 2 * prebuffer_ms / 1000)
        self.stream = sd.RawOutputStream(
            samplerate=samplerate, channels=1, dtype="int16",
            blocksize=480, latency="high", callback=self._cb,
        )

    def _avail(self):
        return len(self.buf) - self.pos

    def pending_bytes(self):
        with self.lock:
            return self._avail()

    def _cb(self, outdata, frames, time, status):
        need = len(outdata)
        with self.lock:
            if not self.playing:
                if self._avail() < self.prebuffer:
                    outdata[:] = b"\x00" * need
                    return
                self.playing = True
            n = min(need, self._avail())
            outdata[:n] = bytes(self.buf[self.pos:self.pos + n])
            self.pos += n
            if n < need:
                outdata[n:] = b"\x00" * (need - n)
                if self._avail() == 0:
                    self.playing = False
            if self.pos > 2_000_000:
                del self.buf[:self.pos]
                self.pos = 0

    def add(self, pcm: bytes):
        with self.lock:
            self.buf.extend(pcm)

    def clear(self):
        with self.lock:
            del self.buf[:]
            self.pos = 0
            self.playing = False

    def start(self):
        self.stream.start()

    def close(self):
        try:
            self.stream.stop()
        finally:
            self.stream.close()


MIC_RESUME_GUARD_S = 0.4   # 스피커 버퍼가 빈 뒤 잔향 가드(에코 차단)
RENEW_BEFORE_S = 60        # 만료 몇 초 전에 갱신을 시도할지

reset_portaudio()

client = AsyncOpenAI(
    base_url=AZURE_OPENAI_URI.replace("https://", "wss://").rstrip("/") + "/openai/v1",
    api_key=AZURE_OPENAI_API_KEY,
)

# 세션 설정을 변수로 분리 → 최초 적용과 연장 재전송에서 동일하게 재사용
session_config = {
    "type": "realtime",
    "instructions": "너는 삼성전자 Customer Service를 담당하는 음성 기반 AI 상담사야. 사용자와 실시간 음성 대화를 하는 환경에서 모든 응답은 4문장을 넘지 않게 말하고, 실제 사람이 말하듯 자연스럽고 간결하게 표현해",
    "output_modalities": ["audio"],
    "audio": {
        "input": {
            "format": {"type": "audio/pcm", "rate": 24000},
            "transcription": {"model": "gpt-4o-transcribe", "language": "ko"},
            "turn_detection": {
                "type": "server_vad",
                "threshold": 0.5,
                "prefix_padding_ms": 300,
                "silence_duration_ms": 500,
                "create_response": True,
                "interrupt_response": True,
            },
        },
        "output": {
            "voice": "marin",
            "format": {"type": "audio/pcm", "rate": 24000},
        },
    },
}

speaker, mic, sender, keepalive = None, None, None, None
try:
    speaker = Speaker()
    loop = asyncio.get_running_loop()
    mic_q: asyncio.Queue[bytes] = asyncio.Queue()

    def mic_cb(indata, frames, time, status):
        loop.call_soon_threadsafe(mic_q.put_nowait, bytes(indata))

    mic = sd.RawInputStream(
        samplerate=24000, channels=1, dtype="int16", blocksize=0, callback=mic_cb,
    )

    async with client.realtime.connect(model=AZURE_OPENAI_REALTIME_DEPLOYMENT) as conn:
        await conn.session.update(session=session_config)
        speaker.start()
        mic.start()
        print("🎙️  말해보세요. (session.update keep-alive 로 세션 연장 시도)\n")
        print("ℹ️  스피커 데모용 에코 차단 모드: AI 가 말하는 동안엔 마이크가 잠시 멈춥니다.\n")

        # 세션 만료 시각(Unix epoch, 초). session.created/updated 에서 갱신된다.
        session_state = {"expires_at": None}

        # 만료 RENEW_BEFORE_S 초 전에 session.update 를 재전송해 연장을 시도한다.
        async def keep_session_alive():
            while True:
                exp = session_state["expires_at"]
                if exp is None:
                    await asyncio.sleep(1)
                    continue
                wait = exp - time.time() - RENEW_BEFORE_S
                if wait > 0:
                    await asyncio.sleep(wait)
                left = int(session_state["expires_at"] - time.time()) if session_state["expires_at"] else 0
                print(f"\n♻️  세션 만료 {left}s 전 → session.update 재전송(연장 시도)")
                session_state["expires_at"] = None   # 새 expires_at 받을 때까지 재전송 방지
                try:
                    await conn.session.update(session=session_config)
                except Exception as e:
                    print(f"\n(세션 갱신 경고: {e})")
                await asyncio.sleep(2)   # 폭주 방지

        keepalive = asyncio.create_task(keep_session_alive())

        mic_gate = {"resume_at": 0.0}

        async def pump_mic():
            while True:
                chunk = await mic_q.get()
                now = time.monotonic()
                if speaker.pending_bytes() > 0:
                    mic_gate["resume_at"] = now + MIC_RESUME_GUARD_S
                    continue
                if now < mic_gate["resume_at"]:
                    continue
                await conn.input_audio_buffer.append(
                    audio=base64.b64encode(chunk).decode(),
                )

        sender = asyncio.create_task(pump_mic())
        async for event in conn:
            if event.type == "session.created":
                session_state["expires_at"] = getattr(event.session, "expires_at", None)
                exp = session_state["expires_at"]
                left = int(exp - time.time()) if exp else None
                print(f"✅ 세션 생성 완료 (만료까지 {left}s)" if left else "✅ 세션 생성 완료")
            elif event.type == "session.updated":
                new_exp = getattr(event.session, "expires_at", None)
                if new_exp is not None:
                    prev = session_state["expires_at"]
                    session_state["expires_at"] = new_exp
                    if prev is None:   # 연장 재전송에 대한 응답으로 도착한 경우
                        left = int(new_exp - time.time())
                        print(f"🔁 세션 갱신됨 (만료까지 {left}s)")
            if event.type == "response.output_audio.delta":
                speaker.add(base64.b64decode(event.delta))
            elif event.type == "response.output_audio_transcript.delta":
                print(event.delta, end="", flush=True)
            elif event.type == "conversation.item.input_audio_transcription.completed":
                print(f"\n🧑 {event.transcript}")
            elif event.type == "conversation.item.input_audio_transcription.failed":
                print("\n❌ 전사 실패:", event.error)
            elif event.type == "input_audio_buffer.speech_started":
                print("\n🎧 음성 인식 시작...")
                speaker.clear()
            elif event.type == "response.done":
                print("\n✅ 응답 완료")
                print()
            elif event.type == "error":
                print("\n⚠️  ERROR:", event.error)
finally:
    if keepalive:
        keepalive.cancel()
    if sender:
        sender.cancel()
    if mic:
        try:
            mic.stop(); mic.close()
        except Exception as e:
            print(f"마이크 종료 에러: {e}")
    if speaker:
        try:
            speaker.close()
        except Exception as e:
            print(f"스피커 종료 에러: {e}")

🎙️  말해보세요. (session.update keep-alive 로 세션 연장 시도)

ℹ️  스피커 데모용 에코 차단 모드: AI 가 말하는 동안엔 마이크가 잠시 멈춥니다.

✅ 세션 생성 완료 (만료까지 3599s)


In [ ]:
import asyncio
import base64
import threading
import time

import sounddevice as sd
from openai import AsyncOpenAI

# ══════════════════════════════════════════════════════════════════════════
# 방식 ② 세션 타임아웃 대응: "재연결 + 컨텍스트 복원" (Azure 권장)
# ──────────────────────────────────────────────────────────────────────────
#   ① 대화 중 user/assistant transcript(텍스트)를 history 에 계속 저장
#   ② 만료 시각이 되면 "완전한 idle 순간"까지 기다렸다가 재연결(→ 발화가 안 끊김)
#        완전한 idle = AI 응답 종료 + 사용자 발화 종료 + 스피커 버퍼 소진
#   ③ 새 세션에서 session.update 후, conversation.item.create 로 history 재주입
# ※ 오디오는 재주입 불가(텍스트만 복원 가능). user=input_text, assistant=output_text.
# ══════════════════════════════════════════════════════════════════════════

# 재실행 시 남아 있는 PortAudio 상태 초기화 (-9986 예방)
def reset_portaudio():
    try:
        sd._terminate()
    except Exception as e:
        print(f"PortAudio reset error: {e}")
    sd._initialize()

# 지터 버퍼 스피커 (재생 끊김 완화)
class Speaker:
    def __init__(self, samplerate=24000, prebuffer_ms=300):
        self.sr = samplerate
        self.buf = bytearray()
        self.pos = 0
        self.lock = threading.Lock()
        self.playing = False
        self.prebuffer = int(samplerate * 2 * prebuffer_ms / 1000)
        self.stream = sd.RawOutputStream(
            samplerate=samplerate, channels=1, dtype="int16",
            blocksize=480, latency="high", callback=self._cb,
        )

    def _avail(self):
        return len(self.buf) - self.pos

    def pending_bytes(self):
        with self.lock:
            return self._avail()

    def _cb(self, outdata, frames, time, status):
        need = len(outdata)
        with self.lock:
            if not self.playing:
                if self._avail() < self.prebuffer:
                    outdata[:] = b"\x00" * need
                    return
                self.playing = True
            n = min(need, self._avail())
            outdata[:n] = bytes(self.buf[self.pos:self.pos + n])
            self.pos += n
            if n < need:
                outdata[n:] = b"\x00" * (need - n)
                if self._avail() == 0:
                    self.playing = False
            if self.pos > 2_000_000:
                del self.buf[:self.pos]
                self.pos = 0

    def add(self, pcm: bytes):
        with self.lock:
            self.buf.extend(pcm)

    def clear(self):
        with self.lock:
            del self.buf[:]
            self.pos = 0
            self.playing = False

    def start(self):
        self.stream.start()

    def close(self):
        try:
            self.stream.stop()
        finally:
            self.stream.close()


MIC_RESUME_GUARD_S = 0.4       # 스피커 버퍼가 빈 뒤 잔향 가드(에코 차단)
RECONNECT_BEFORE_S = 30        # (실전) 만료 몇 초 전에 재연결 준비를 시작할지
RECONNECT_HARD_MARGIN_S = 3    # (실전) expires_at 이 이만큼 남으면 idle 을 못 기다리고 강제 재연결
# ── 테스트 스위치 ──────────────────────────────────────────────────────
# 값이 정해져 있으면 실제 expires_at 을 무시하고, "세션 시작 후 이 초만큼" 지나면
# (완전한 idle 순간에) 재연결한다. 동작을 빨리 확인하기 위한 용도.
# 실전에서는 None 으로 두어 expires_at 기준으로 동작하게 한다.
TEST_RECONNECT_AFTER_S = 60    # 예: 1분 뒤 재연결 테스트 / 실전은 None

reset_portaudio()

client = AsyncOpenAI(
    base_url=AZURE_OPENAI_URI.replace("https://", "wss://").rstrip("/") + "/openai/v1",
    api_key=AZURE_OPENAI_API_KEY,
)

# 세션 설정: 최초 연결과 재연결에서 동일하게 재사용
session_config = {
    "type": "realtime",
    "instructions": "너는 삼성전자 Customer Service를 담당하는 음성 기반 AI 상담사야. 사용자와 실시간 음성 대화를 하는 환경에서 모든 응답은 4문장을 넘지 않게 말하고, 실제 사람이 말하듯 자연스럽고 간결하게 표현해",
    "output_modalities": ["audio"],
    "audio": {
        "input": {
            "format": {"type": "audio/pcm", "rate": 24000},
            "transcription": {"model": "gpt-4o-transcribe", "language": "ko"},
            "turn_detection": {
                "type": "server_vad",
                "threshold": 0.5,
                "prefix_padding_ms": 300,
                "silence_duration_ms": 500,
                "create_response": True,
                "interrupt_response": True,
            },
        },
        "output": {
            "voice": "marin",
            "format": {"type": "audio/pcm", "rate": 24000},
        },
    },
}

# 재연결 간 보존되는 대화 컨텍스트(텍스트 transcript). 오디오는 복원 불가.
history = []   # [{"role": "user"|"assistant", "text": str}]

# 저장해둔 history 를 새 세션에 재주입 → 이전 맥락 복원
async def inject_history(conn, history):
    for turn in history:
        # 역할별 content type 이 다르다: user=input_text, assistant=output_text
        # (GA realtime API 는 assistant 재주입에 output_text 를 요구한다)
        content = ([{"type": "input_text", "text": turn["text"]}]
                   if turn["role"] == "user"
                   else [{"type": "output_text", "text": turn["text"]}])
        await conn.conversation.item.create(item={
            "type": "message", "role": turn["role"], "content": content,
        })

speaker, mic, sender = None, None, None
try:
    speaker = Speaker()
    loop = asyncio.get_running_loop()
    mic_q: asyncio.Queue[bytes] = asyncio.Queue()

    def mic_cb(indata, frames, time, status):
        loop.call_soon_threadsafe(mic_q.put_nowait, bytes(indata))

    mic = sd.RawInputStream(
        samplerate=24000, channels=1, dtype="int16", blocksize=0, callback=mic_cb,
    )
    speaker.start()
    mic.start()
    mode = f"테스트: {TEST_RECONNECT_AFTER_S}s 후 재연결" if TEST_RECONNECT_AFTER_S else "실전: expires_at 기준"
    print(f"🎙️  말해보세요. (세션 만료 시 대화가 비는 순간 재연결 + 맥락 복원 · {mode})\n")
    print("ℹ️  스피커 데모용 에코 차단 모드: AI 가 말하는 동안엔 마이크가 잠시 멈춥니다.\n")

    mic_gate = {"resume_at": 0.0}      # 마이크 게이팅(에코 차단) — 연결 간 유지
    cur_assistant = {"text": ""}       # 진행 중 assistant 응답 transcript 누적
    # 재연결 타이밍 판단용 상태: AI 발화 중 / 사용자 발화 중
    state = {"active": False, "user_speaking": False}
    session_no = 0

    # ── 재연결 루프 ────────────────────────────────────────────────────
    # 커널 인터럽트(중단)로 빠져나올 때까지, 세션이 만료되면 새 세션으로 이어간다.
    while True:
        session_no += 1
        async with client.realtime.connect(model=AZURE_OPENAI_REALTIME_DEPLOYMENT) as conn:
            await conn.session.update(session=session_config)
            if history:
                await inject_history(conn, history)   # ③ 이전 맥락 복원
                print(f"🔁 컨텍스트 복원: {len(history)}턴 재주입")

            # 마이크 전송 태스크(현재 conn 에 종속) — 연결이 닫히면 조용히 종료
            async def pump_mic(conn=conn):
                while True:
                    chunk = await mic_q.get()
                    now = time.monotonic()
                    if speaker.pending_bytes() > 0:
                        mic_gate["resume_at"] = now + MIC_RESUME_GUARD_S
                        continue
                    if now < mic_gate["resume_at"]:
                        continue
                    try:
                        await conn.input_audio_buffer.append(
                            audio=base64.b64encode(chunk).decode())
                    except Exception:
                        return   # 재연결 중 연결이 닫힘 → 태스크 종료
            sender = asyncio.create_task(pump_mic())

            # 만료 감시 태스크
            #  - (테스트) 고정 시간 뒤 / (실전) expires_at - RECONNECT_BEFORE_S 에 깨어난다.
            #  - 깨어난 뒤 "완전한 idle 순간"까지 기다렸다가 conn.close() → 발화가 안 잘린다.
            #    완전한 idle = AI 응답 비활성 + 사용자 발화 중 아님 + 스피커 버퍼 빔
            #  - 단, 실전에서 expires_at 하드 리밋이 임박하면(RECONNECT_HARD_MARGIN_S) 강제로 닫는다.
            watchdog = {"task": None}
            async def watch_expiry(expires_at, conn=conn):
                if TEST_RECONNECT_AFTER_S is not None:
                    wait = TEST_RECONNECT_AFTER_S
                    hard_deadline = None
                else:
                    wait = expires_at - time.time() - RECONNECT_BEFORE_S
                    hard_deadline = expires_at - RECONNECT_HARD_MARGIN_S
                if wait > 0:
                    await asyncio.sleep(wait)
                print("\n♻️  세션 만료 임박 → 대화가 비는 순간에 재연결합니다...")
                # AI 발화 중 · 사용자 발화 중 · 스피커 재생 잔량이 있으면 끝날 때까지 대기
                while state["active"] or state["user_speaking"] or speaker.pending_bytes() > 0:
                    if hard_deadline is not None and time.time() >= hard_deadline:
                        print("\n⏳ (만료 하드 리밋 임박 — idle 을 다 기다리지 못하고 재연결)")
                        break
                    await asyncio.sleep(0.2)
                try:
                    await conn.close()
                except Exception:
                    pass

            try:
                async for event in conn:
                    if event.type == "session.created":
                        state["active"] = False
                        state["user_speaking"] = False
                        exp = getattr(event.session, "expires_at", None)
                        left = int(exp - time.time()) if exp else None
                        print(f"✅ 세션 #{session_no} 시작"
                              + (f" (만료까지 {left}s)" if left else ""))
                        # 테스트 모드면 expires_at 이 없어도 watchdog 을 띄운다
                        if watchdog["task"] is None and (exp or TEST_RECONNECT_AFTER_S is not None):
                            watchdog["task"] = asyncio.create_task(watch_expiry(exp))
                    # AI 응답 시작 → 발화 진행 중으로 표시
                    elif event.type == "response.created":
                        state["active"] = True
                    elif event.type == "response.output_audio.delta":
                        speaker.add(base64.b64decode(event.delta))
                    # ① assistant transcript 누적 (재주입용)
                    elif event.type == "response.output_audio_transcript.delta":
                        cur_assistant["text"] += event.delta
                        print(event.delta, end="", flush=True)
                    # ① user transcript 저장 (재주입용)
                    elif event.type == "conversation.item.input_audio_transcription.completed":
                        history.append({"role": "user", "text": event.transcript})
                        print(f"\n🧑 {event.transcript}")
                    elif event.type == "conversation.item.input_audio_transcription.failed":
                        print("\n❌ 전사 실패:", event.error)
                    # 사용자가 말하기 시작 → 발화 중으로 표시(+끼어들기 시 재생 중단)
                    elif event.type == "input_audio_buffer.speech_started":
                        state["user_speaking"] = True
                        print("\n🎧 음성 인식 시작...")
                        speaker.clear()
                    # 사용자가 말하기 종료 → 발화 종료로 표시
                    elif event.type == "input_audio_buffer.speech_stopped":
                        state["user_speaking"] = False
                    elif event.type == "response.done":
                        if cur_assistant["text"].strip():
                            history.append({"role": "assistant", "text": cur_assistant["text"]})
                        cur_assistant["text"] = ""
                        state["active"] = False   # AI 발화 종료
                        print("\n✅ 응답 완료\n")
                    elif event.type == "error":
                        print("\n⚠️  ERROR:", event.error)
            except Exception as e:
                # 연결 종료/네트워크 오류 → 바깥 while 이 재연결
                print(f"\n(연결 종료: {e}) 재연결합니다...")
            finally:
                if watchdog["task"]:
                    watchdog["task"].cancel()
                if sender:
                    sender.cancel()
            # 다음 세션에서 새로 재생할 것이므로 잔여 오디오만 정리(idle 순간이라 잘릴 게 없음)
            speaker.clear()
finally:
    if sender:
        sender.cancel()
    if mic:
        try:
            mic.stop(); mic.close()
        except Exception as e:
            print(f"마이크 종료 에러: {e}")
    if speaker:
        try:
            speaker.close()
        except Exception as e:
            print(f"스피커 종료 에러: {e}")